In [46]:
import os
import sys
from typing import Optional, List, Dict

import pandas as pd


ANALYSIS_DIR = os.path.join("data", "analysis")
DERIVED_DIR = "data_derived"


def log(msg: str) -> None:
    print(msg, flush=True)


def pick_name_col(df: pd.DataFrame) -> Optional[str]:
    for c in ["Name", "NAME", "neighborhood", "Neighborhood", "neighborhood_name"]:
        if c in df.columns:
            return c
    return None


def normalize_key_series(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.casefold()
    )


def safe_read_csv(path: str) -> pd.DataFrame:
    try:
        return pd.read_csv(path, low_memory=False)
    except Exception as e:
        raise SystemExit(f"Failed to read {path}: {e}")


def suffix_columns(df: pd.DataFrame, exclude: List[str], suffix: str) -> pd.DataFrame:
    ren: Dict[str, str] = {}
    for c in df.columns:
        if c in exclude:
            continue
        ren[c] = f"{c}{suffix}"
    return df.rename(columns=ren)


def main() -> None:
    houses_csv = os.path.join(ANALYSIS_DIR, "servicelines_with_imputed_materials.csv")
    nbh_csv = os.path.join(DERIVED_DIR, "neighborhoods_from_matches_combined.csv")
    out_csv = os.path.join("data", "regression", "servicelines_lead_logits_dataset.csv")

    if not os.path.exists(houses_csv):
        raise SystemExit(f"Input not found: {houses_csv}")
    if not os.path.exists(nbh_csv):
        raise SystemExit(f"Input not found: {nbh_csv} (run build_neighborhood_from_matches.py)")

    os.makedirs(os.path.dirname(out_csv), exist_ok=True)

    houses = safe_read_csv(houses_csv)
    nbh = safe_read_csv(nbh_csv)

    # Resolve join keys
    nbh_key = pick_name_col(nbh)
    house_key = pick_name_col(houses)
    if nbh_key is None:
        raise SystemExit("Could not find a neighborhood name column in neighborhoods_from_matches_combined.csv")
    if house_key is None:
        raise SystemExit("Could not find a neighborhood name column in servicelines_with_imputed_materials.csv")

    # Prepare right-hand side columns with _neighborhood suffix
    nbh_renamed = suffix_columns(nbh.copy(), exclude=[nbh_key], suffix="_neighborhood")

    # Normalize keys for a robust merge (case/space insensitive)
    houses["__join_key__"] = normalize_key_series(houses[house_key])
    nbh_renamed["__join_key__"] = normalize_key_series(nbh[nbh_key])

    merged = houses.merge(
        nbh_renamed.drop(columns=[nbh_key]),
        on="__join_key__",
        how="left",
        suffixes=("", "_neighborhood"),
    )

    # Clean up temp key and, if useful, keep the original house neighborhood name
    merged = merged.drop(columns=["__join_key__"]).copy()

    merged.to_csv(out_csv, index=False)
    log(f"[OK] Wrote merged logits dataset -> {out_csv}")


if __name__ == "__main__":
    main()



[OK] Wrote merged logits dataset -> data\regression\servicelines_lead_logits_dataset.csv


In [47]:
# Imports + load minimalist EDA file
import pandas as pd
from pathlib import Path
import numpy as np
path = Path("data/regression/servicelines_lead_logits_dataset.csv")
df = pd.read_csv(path)

C:\Users\bradk\AppData\Local\Temp\ipykernel_8996\1326226116.py:6: DtypeWarning: Columns (31,32,33,34,39) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


In [48]:
df.columns.tolist()

['accountid',
 'address',
 'location',
 'sensitivepop',
 'disadvantaged',
 'utilassetid',
 'utilmaterial',
 'everlead',
 'utilinstalldate',
 'utildiameter',
 'utilsource',
 'utilverified',
 'utilverifmethod',
 'utilverifdate',
 'utilstatus',
 'utilnotes',
 'custassetid',
 'custmaterial',
 'custinstalldate',
 'custdiameter',
 'custsource',
 'custverified',
 'custverifmethod',
 'custverifdate',
 'custstatus',
 'custnotes',
 'bothsidesstatus',
 'leadconnector',
 'leadsolder',
 'otherfittings',
 'buildingtype',
 'pointofentry',
 'copperwithlead',
 'samplingsite',
 'replacestatus',
 'scheddate',
 'utilreplacedate',
 'custscheddate',
 'custreplacedate',
 'replacereason',
 'custnotified',
 'notifydate',
 'yearstructbuilt',
 'Creator',
 'CreationDate',
 'Editor',
 'EditDate',
 'GlobalID',
 'OBJECTID',
 'Latitude',
 'Longitude',
 'BasisOfAdditionToInventory',
 'utilsource_detailed',
 'custsource_detailed',
 'LCRRSampleTier',
 'ServiceType',
 'PremiseCode',
 'ZipCode',
 'ServiceNumber',
 'ModelS

In [49]:
from datetime import datetime

# drop missing years
df = df.dropna(subset=["yearstructbuilt"])
# drop years = "0"
df = df[df["yearstructbuilt"] > 0]
# drop future years
df = df[df["yearstructbuilt"] < 2026]
# bin by decade (or any bin width you like). 
df["year_bin"] = (df["yearstructbuilt"] // 10) * 10   # decade bins
#group all years before 1800
df.loc[df["yearstructbuilt"] < 1800, "year_bin"] = 1800
# Lead ban indicator: 0 = pre-1988, 1 = 1988 or later
df["post_lead_ban"] = (df["yearstructbuilt"] >= 1988).astype(int)
# Building age (in years, as of current year)
current_year = datetime.now().year
df["building_age"] = current_year - df["yearstructbuilt"]
df["building_age_decades"] = df["building_age"] // 10

# drop missing
df_model = df.dropna(subset=["bothsidesstatus_imputed", "Pblack", "householdincome_avghinc_cy", "year_bin"]).copy()

# scale income to 10k units for interpretability
df_model["MHI_10k"] = df_model["householdincome_avghinc_cy"] / 10000

# binary dependent variable
df_model["lead_flag"] = (df_model["bothsidesstatus_imputed"] == "Lead/GRR").astype(int)
df_model["cust_lead_flag"] = (df_model["custmaterial_cat_imputed"] == "Lead/GRR").astype(int)
df_model["util_lead_flag"] = (df_model["utilmaterial_cat_imputed"] == "Lead/GRR").astype(int)

# scale income to 10k units for interpretability
df_model["MHI_10k_neighborhood"] = df_model["householdincome_avghinc_cy_neighborhood"] / 10000


In [ ]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# define formula: lead probability ~ race, income, and year_bin (categorical)
formula = "lead_flag ~ Pblack + MHI_10k + C(year_bin)"

# fit with cluster-robust SEs
logit_clustered = smf.logit(formula=formula, data=df_model).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_model["neighborhood_name"]}
)

# summarize results
print(logit_clustered.summary())

# odds ratios for readability
odds_ratios = pd.DataFrame({
    "term": logit_clustered.params.index,
    "coef": logit_clustered.params.values,
    "odds_ratio": logit_clustered.params.apply(lambda x: np.exp(x)),
    "p_value": logit_clustered.pvalues,
})
print("\nOdds Ratios:")
print(odds_ratios)


Optimization terminated successfully.
         Current function value: 0.432878
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:              lead_flag   No. Observations:                80374
Model:                          Logit   Df Residuals:                    80349
Method:                           MLE   Df Model:                           24
Date:                Tue, 28 Oct 2025   Pseudo R-squ.:                  0.3602
Time:                        15:39:50   Log-Likelihood:                -34792.
converged:                       True   LL-Null:                       -54376.
Covariance Type:              cluster   LLR p-value:                     0.000
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                -1.4209      0.799     -1.778      0.075      -2.987       0.

In [ ]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# drop missing
df_model = df.dropna(subset=["bothsidesstatus_imputed", "pct_black_neighborhood", "householdincome_avghinc_cy_neighborhood", "year_bin"]).copy()



# define formula: lead probability ~ race, income, and year_bin (categorical)
formula = "lead_flag ~ pct_black_neighborhood + MHI_10k_neighborhood + C(year_bin)"

# fit with cluster-robust SEs
logit_clustered = smf.logit(formula=formula, data=df_model).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_model["neighborhood_name"]}
)

# summarize results
print(logit_clustered.summary())

# odds ratios for readability
odds_ratios = pd.DataFrame({
    "term": logit_clustered.params.index,
    "coef": logit_clustered.params.values,
    "odds_ratio": logit_clustered.params.apply(lambda x: np.exp(x)),
    "p_value": logit_clustered.pvalues,
})
print("\nOdds Ratios:")
print(odds_ratios)


Optimization terminated successfully.
         Current function value: 0.433753
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:              lead_flag   No. Observations:                80374
Model:                          Logit   Df Residuals:                    80349
Method:                           MLE   Df Model:                           24
Date:                Tue, 28 Oct 2025   Pseudo R-squ.:                  0.3589
Time:                        15:41:15   Log-Likelihood:                -34862.
converged:                       True   LL-Null:                       -54376.
Covariance Type:              cluster   LLR p-value:                     0.000
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 -1.3689      0.880     -1.556      0.120      -3.094      

In [16]:
# fit sans cluster
logit = smf.logit(formula=formula, data=df_model).fit()

# summarize results
print(logit.summary())

# odds ratios for readability
odds_ratios = pd.DataFrame({
    "term": logit.params.index,
    "coef": logit.params.values,
    "odds_ratio": logit.params.apply(lambda x: np.exp(x)),
    "p_value": logit.pvalues,
})
print("\nOdds Ratios:")
print(odds_ratios)


Optimization terminated successfully.
         Current function value: 0.433753
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:              lead_flag   No. Observations:                80374
Model:                          Logit   Df Residuals:                    80349
Method:                           MLE   Df Model:                           24
Date:                Tue, 28 Oct 2025   Pseudo R-squ.:                  0.3589
Time:                        15:44:21   Log-Likelihood:                -34862.
converged:                       True   LL-Null:                       -54376.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 -1.3689      0.382     -3.580      0.000      -2.118      

In [ ]:
#Adding disadvantaged

# define formula: lead probability ~ race, income, year_bin (categorical), disadvantaged
formula = "lead_flag ~ pct_black_neighborhood + MHI_10k_neighborhood + C(year_bin) + C(disadvantaged)"

logit = smf.logit(formula=formula, data=df_model).fit()

# summarize results
print(logit.summary())

# odds ratios for readability
odds_ratios = pd.DataFrame({
    "term": logit.params.index,
    "coef": logit.params.values,
    "odds_ratio": logit.params.apply(lambda x: np.exp(x)),
    "p_value": logit.pvalues,
})
print("\nOdds Ratios:")
print(odds_ratios)


Optimization terminated successfully.
         Current function value: 0.433580
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:              lead_flag   No. Observations:                80374
Model:                          Logit   Df Residuals:                    80348
Method:                           MLE   Df Model:                           25
Date:                Tue, 28 Oct 2025   Pseudo R-squ.:                  0.3591
Time:                        17:01:25   Log-Likelihood:                -34849.
converged:                       True   LL-Null:                       -54376.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                  -1.3371      0.382     -3.498      0.000      -2.086   

In [ ]:
#Adding disadvantaged

# define formula: lead probability ~ race, income, year_bin (categorical), disadvantaged
formula = "lead_flag ~ pct_black_neighborhood + MHI_10k_neighborhood + C(year_bin) + C(disadvantaged)"

logit = smf.logit(formula=formula, data=df_model).fit()

# summarize results
print(logit.summary())

# odds ratios for readability
odds_ratios = pd.DataFrame({
    "term": logit.params.index,
    "coef": logit.params.values,
    "odds_ratio": logit.params.apply(lambda x: np.exp(x)),
    "p_value": logit.pvalues,
})
print("\nOdds Ratios:")
print(odds_ratios)


In [24]:
#Adding more characteristics

# define formula: lead probability ~ race, income, year_bin (categorical), disadvantaged, buildingtype
formula = "lead_flag ~ MHI_10k_neighborhood + pct_black_neighborhood + C(year_bin) + C(disadvantaged) + C(buildingtype)"

logit = smf.logit(formula=formula, data=df_model).fit()

# summarize results
print(logit.summary())

# odds ratios for readability
odds_ratios = pd.DataFrame({
    "term": logit.params.index,
    "coef": logit.params.values,
    "odds_ratio": logit.params.apply(lambda x: np.exp(x)),
    "p_value": logit.pvalues,
})
print("\nOdds Ratios:")
print(odds_ratios)


Optimization terminated successfully.
         Current function value: 0.415712
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:              lead_flag   No. Observations:                80374
Model:                          Logit   Df Residuals:                    80345
Method:                           MLE   Df Model:                           28
Date:                Tue, 28 Oct 2025   Pseudo R-squ.:                  0.3855
Time:                        17:43:37   Log-Likelihood:                -33412.
converged:                       True   LL-Null:                       -54376.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                   coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------------------
Intercept                               

In [41]:
#Lead ban indicator and building age

formula = "lead_flag ~ pct_black_neighborhood + MHI_10k_neighborhood + building_age + post_lead_ban"

logit = smf.logit(formula=formula, data=df_model).fit()

# summarize results
print(logit.summary())

# odds ratios for readability
odds_ratios = pd.DataFrame({
    "term": logit.params.index,
    "coef": logit.params.values,
    "odds_ratio": logit.params.apply(lambda x: np.exp(x)),
    "p_value": logit.pvalues,
})
print("\nOdds Ratios:")
print(odds_ratios)

Optimization terminated successfully.
         Current function value: 0.531412
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:              lead_flag   No. Observations:                80374
Model:                          Logit   Df Residuals:                    80369
Method:                           MLE   Df Model:                            4
Date:                Wed, 29 Oct 2025   Pseudo R-squ.:                  0.2145
Time:                        09:14:40   Log-Likelihood:                -42712.
converged:                       True   LL-Null:                       -54376.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 -3.8515      0.061    -62.975      0.000      -3.971      

In [54]:
#Lead ban indicator and building age interaction

formula = "lead_flag ~ pct_black_neighborhood + MHI_10k_neighborhood + building_age * post_lead_ban"

logit = smf.logit(formula=formula, data=df_model).fit()

# summarize results
print(logit.summary())

# odds ratios for readability
odds_ratios = pd.DataFrame({
    "term": logit.params.index,
    "coef": logit.params.values,
    "odds_ratio": logit.params.apply(lambda x: np.exp(x)),
    "p_value": logit.pvalues,
})
print("\nOdds Ratios:")
print(odds_ratios)

Optimization terminated successfully.
         Current function value: 0.522617
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:              lead_flag   No. Observations:                80374
Model:                          Logit   Df Residuals:                    80368
Method:                           MLE   Df Model:                            5
Date:                Wed, 29 Oct 2025   Pseudo R-squ.:                  0.2275
Time:                        09:25:07   Log-Likelihood:                -42005.
converged:                       True   LL-Null:                       -54376.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept                     -4.0806      0.062    -65.577      0.000      